In [73]:
import pandas as pd
import requests
import notebook
import sqlalchemy
import psycopg2
import json
import numpy as np
 

Pull key froms json

In [59]:
with open("keys.json", "r") as f:
    keys = json.load(f)
    API_KEY = keys.get("API_KEY")


  

Header and URL

In [60]:
BASE_URL = "https://api.collegebasketballdata.com/"
HEADERS = {'Authorization': f'Bearer {API_KEY}', 'accept': 'application/json'}

Getting dataframe in range of seasons

In [ ]:
# Date Range (2019-2026)
Seasons = range(2019, 2026)


# Fetch Data
def fetch_to_dataframe(endpoint, season):
    url = f"{BASE_URL}/{endpoint}"
    params = {"season": season}
    try:
        response = requests.get(url, headers=HEADERS, params=params)
        if response.status_code == 200:
            data = response.json()
            if data and len(data) > 0:
                return pd.DataFrame(data)
            else:
                return pd.DataFrame()
        else:
            print(f' Error {response.status_code}: {endpoint}')
            return pd.DataFrame()
    except Exception as e:
        print(f' Exception: {e}')
        return pd.DataFrame()  

empty data sets

In [62]:
rankings_df = pd.DataFrame()
elo_df = pd.DataFrame()
team_stats_df = pd.DataFrame()
games_df = pd.DataFrame()

rankings_df

In [124]:
for season in Seasons:
    print(f"\nProcessing season: {season}")

    df = fetch_to_dataframe("rankings", season)
    if not df.empty:
        rankings_df = pd.concat([rankings_df, df], ignore_index=True)
        print(f"Rankings: {len(df)} rows")


Processing season: 2019
Rankings: 1650 rows

Processing season: 2020
Rankings: 1729 rows

Processing season: 2021
Rankings: 1493 rows

Processing season: 2022
Rankings: 1585 rows

Processing season: 2023
Rankings: 1668 rows

Processing season: 2024
Rankings: 1825 rows

Processing season: 2025
Rankings: 1843 rows


In [126]:
len(rankings_df)

23586

In [127]:
print(rankings_df.shape)
print(rankings_df.dtypes)

(23586, 11)
season               int64
seasonType             str
week                 int64
pollDate               str
pollType               str
teamId               int64
team                   str
conference             str
ranking            float64
points               int64
firstPlaceVotes      int64
dtype: object


elo_df

In [130]:
df = fetch_to_dataframe("ratings/elo", season)
if not df.empty:
    elo_df = pd.concat([elo_df, df], ignore_index=True)
    print(f" Elo ratings: {len(df)} rows")
else:
    print(" Elo ratings: No data")

 Elo ratings: 364 rows


In [129]:
print(elo_df.shape)
print(elo_df.dtypes)

(728, 5)
season        int64
teamId        int64
team            str
conference      str
elo           int64
dtype: object


team_stats_df

In [131]:
df = fetch_to_dataframe("stats/team/season", season)
if not df.empty:
    team_stats_df = pd.concat([team_stats_df, df], ignore_index=True)
    print(f" Team Stats: {len(df)} rows")
else:
    print(" Team Stats: No data")

 Team Stats: 700 rows


In [132]:
print(team_stats_df.shape)
print(team_stats_df.dtypes)

(1400, 12)
season             int64
seasonLabel          str
teamId             int64
team                 str
conference           str
games              int64
wins               int64
losses             int64
totalMinutes       int64
pace             float64
teamStats         object
opponentStats     object
dtype: object


games_df

In [133]:
len(games_df)


72000

In [136]:
season

2025

In [137]:
for season in Seasons:
    print(f"\nProcessing season: {season}")
    df = fetch_to_dataframe("games", season)
    if not df.empty:
        games_df = pd.concat([games_df, df], ignore_index=True)
        print(f" Games: {len(df)} rows")
    else:
        print(" Games: No data")


Processing season: 2019
 Games: 3000 rows

Processing season: 2020
 Games: 3000 rows

Processing season: 2021
 Games: 3000 rows

Processing season: 2022
 Games: 3000 rows

Processing season: 2023
 Games: 3000 rows

Processing season: 2024
 Games: 3000 rows

Processing season: 2025
 Games: 3000 rows


In [138]:
print(games_df.shape)
print(games_df.dtypes)

(93000, 39)
id                    int64
sourceId                str
seasonLabel             str
season                int64
seasonType              str
tournament           object
startDate               str
startTimeTbd           bool
neutralSite            bool
conferenceGame         bool
gameType                str
status                  str
gameNotes               str
attendance          float64
homeTeamId            int64
homeTeam                str
homeConferenceId    float64
homeConference          str
homeSeed             object
homePoints          float64
homePeriodPoints     object
homeWinner           object
homeTeamEloStart    float64
homeTeamEloEnd      float64
awayTeamId            int64
awayTeam                str
awayConferenceId    float64
awayConference          str
awaySeed             object
awayPoints          float64
awayPeriodPoints     object
awayWinner           object
awayTeamEloStart    float64
awayTeamEloEnd      float64
excitement          float64
venueId 

Wins Above Bubble Calculation

In [139]:
print("\n" + "="*60)
print("DIAGNOSTIC: Checking game counts per season")
print("="*60)

for season in Seasons:
    season_games = games_df[games_df['season'] == season]
    print(f"Season {season-1}-{season}: {len(season_games)} games")

# Check if we have the expected number of games
# Expected: ~5,500+ games per full D1 season
print(f"\nExpected ~5,500+ games per full season")
print(f"Actual max games in any season: {games_df['season'].value_counts().max() if not games_df.empty else 0}")


DIAGNOSTIC: Checking game counts per season
Season 2018-2019: 15000 games
Season 2019-2020: 12000 games
Season 2020-2021: 12000 games
Season 2021-2022: 12000 games
Season 2022-2023: 12000 games
Season 2023-2024: 12000 games
Season 2024-2025: 18000 games

Expected ~5,500+ games per full season
Actual max games in any season: 18000


In [ ]:
class WinsAboveBubbleCalculator:

    def __init__(self, bubble_rank_percentile: float = 0.15):
        """ 
        Percentile for bubble cutoff is around a ranking of 50-60 
        top 15% would rank 50-60
        """
    def calculate_win_prob(self, team_elo: float, opponent_elo: float, bubble_elo: float) -> float:

        elo_diff = opponent_elo - bubble_elo
        win_prob = 1.0 / (1.0 + 10 ** (elo_diff / 400))
        return np.clip(win_prob, 0.05, 0.95)
    
    def get_bubble_elo_threshold(self, elo_data: pd.DataFrame, season: int) -> float:

        if 'season' not in elo_data.columns:
            return 1500.0
        season_elo = elo_data[elo_data['season'] == season].copy()
        if season_elo.empty:
            return 1500.0 #default elo
        elo_col = 'elo' if 'elo' in season_elo.columns else 'rating' if 'rating' in season_elo.columns else None
        if elo_col is None:
            return 1500.0
        
        season_elo = season_elo.sort_values(elo_col, ascending=False).reset_index(drop=True)
        bubble_index = min(int(len(season_elo) * self.bubble_rank_percentile),len(season_elo)-1)
        bubble_elo = season_elo.iloc[bubble_index][elo_col]
        return bubble_elo

    def calculate_season_wab(self,season: int, games: pd.DataFrame,elo_data: pd.DataFrame) -> pd.DataFrame:
        season_start = season - 1
        print(f"\n{'='*60}")
        print(f"Calculating WAB for {season_start}-{season} season")
        print(f"{'='*60}")
        if 'season' not in games.columns:
            print(f" Adding season column to games_df")
            season_games = games.copy()
        else:
            

            season_games = games[games['season']== season].copy()

        if 'completed' in season_games.columns:
            season_games = season_games[season_games['completed']== True]
        elif 'status' in season_games.columns:
            season_games = season_games[season_games['status'] == 'Final']
        if season_games.empty:
            print(f"No completed games found for {season}")
            return pd.DataFrame()
        
        season_elo = elo_data[elo_data['season']== season].copy()
        if season_elo.empty:
            print(f"No ELO data found for {season}")
            return pd.DataFrame()
        bubble_elo = self.get_bubble_elo_threshold(elo_data, season)

        elo_col = 'elo' if 'elo' in season_elo.columns else 'rating'
        elo_dict = dict(zip(season_elo['team'], season_elo[elo_col]))

        home_teams = season_games['homeTeam'].unique()
        away_teams = season_games['awayTeam'].unique()
        all_teams = set(home_teams) | set(away_teams)

        results = []

        for team in all_teams:
            team_games = season_games [
                (season_games['homeTeam'] == team)|
                (season_games['awayTeam'] == team)
            ]

            if team_games.empty:
                continue
            actual_wins = 0
            expected_wins = 0
            games_analyzed = 0

            for _, game in team_games.iterrows():
                if game['homeTeam'] == team:
                    oppenent = game['awayTeam']
                    team_score = game.get('homeScore')
                    opp_score = game.get('awayScore')
                else:
                    oppenent = game['homeTeam']
                    team_score = game.get('awayScore')
                    opp_score = game.get('homeScore')

                if pd.isna(team_score) or pd.isna(opp_score):
                    continue

                if team_score > opp_score:
                    actual_wins += 1
                
                oppenent_elo = elo_dict.get(oppenent, 1500)

                win_prob = self.calculate_win_prob(
                    elo_dict.get(team, 1500),
                    oppenent_elo,
                    bubble_elo
                )
                expected_wins += win_prob
                games_analyzed += 1

            if games_analyzed > 0:
                wab = actual_wins - expected_wins
            
                team_elo = elo_dict.get(team, 1500)
                team_rank = season_elo[season_elo['team'] == team][elo_col].rank(ascending=False, method='min').values[0] if team in season_elo['team'].values else None

                results.appened({
                    'season': season,
                    'team': team,
                    'wab': round(wab, 3),
                    'actual_wins': actual_wins,
                    'expected_wins': round(expected_wins, 3),
                    'games_played': games_analyzed,
                    'team_elo': round(team_elo, 1),
                    'team_rank': int(team_rank) if team_rank and not pd.isna(team_rank) else None,
                    'bubble_elo': round(bubble_elo, 1)
                })
            results_df = pd.DataFrame(results)
            if not results_df.empty:
                results_df = results_df.sort_values('wab', ascending=False).reset_index(drop=True)
                results_df.inset(0, 'rank', results_df.index + 1)
            return results_df
            
        

        
    

    
    
    



NameError: name 'games' is not defined

In [115]:
calculator = WinsAboveBubbleCalculator(bubble_rank_percentile=0.15)

all_wab_results = []
for season in Seasons:
    wab_df = calculator.calculate_season_wab(season,games_df,elo_df)
    if not wab_df.empty:
        all_wab_results.append(wab_df) 

        season_start = '2025'
        print(f"\n top 10 teams - {season_start}")
        print("-"*90)
        display_cols = ['rank', 'team', 'wab', 'actual_wins', 'expected_wins', 'team_rank']
        print(wab_df[display_cols].head(10).to_string(index=False))


Calculating WAB for 2018-2019 season
No completed games found for 2019

Calculating WAB for 2019-2020 season
No completed games found for 2020

Calculating WAB for 2020-2021 season
No completed games found for 2021

Calculating WAB for 2021-2022 season
No completed games found for 2022

Calculating WAB for 2022-2023 season
No completed games found for 2023

Calculating WAB for 2023-2024 season
No completed games found for 2024

Calculating WAB for 2024-2025 season
No completed games found for 2025


Calculate WAB for all seasons

In [116]:
calculator = WinsAboveBubbleCalculator(bubble_rank_percentile=0.15)

all_wab_results = []
for season in Seasons:
    wab_df = calculator.calculate_season_wab(season, games_df, elo_df)

    if not wab_df.empty:
        all_wab_results.append(wab_df)
    
        


Calculating WAB for 2018-2019 season
No completed games found for 2019

Calculating WAB for 2019-2020 season
No completed games found for 2020

Calculating WAB for 2020-2021 season
No completed games found for 2021

Calculating WAB for 2021-2022 season
No completed games found for 2022

Calculating WAB for 2022-2023 season
No completed games found for 2023

Calculating WAB for 2023-2024 season
No completed games found for 2024

Calculating WAB for 2024-2025 season
No completed games found for 2025
